# Метод SPD-Faith на MechFaithBench: §4.3, linear probing и activation patching

Этот ноутбук переносит механистический анализ из статьи SPD-Faith (§4.3, наш ноутбук `SPD-Faith Bench.ipynb`)
на новый бенчмарк **MechFaithBench v1.9** и добавляет к нему два метода. Модель та же:
`Qwen/Qwen2-VL-7B-Instruct`.

| метод | что делает | на какой вопрос отвечает |
|---|---|---|
| CosSim (§4.3 статьи) | сравнивает скрытые состояния двух прогонов | «представления разные?» — корреляция |
| **linear probing** | учит логистическую регрессию на скрытых состояниях слоя *l* | «информация *линейно извлекаема* на слое *l*?» |
| **activation patching** | подменяет активации одного прогона активациями другого | «эта позиция/слой *причинно* определяют ответ?» |

## Почему MechFaithBench подходит

1600 **контрфактических пар** `original_image` / `edited_image`: одна контролируемая правка, 8 категорий. У каждой пары:

- `question`: бинарный вопрос с **противоположными** ответами `gold_original` / `gold_edited`. Одна картинка — один
  forward-проход, ответ читаем по логитам «yes»/«no», генерация не нужна;
- маски места правки `roi_evidence_mask_{original,edited}`. По ним видно, какие image-токены содержат правку;
- `diagnostic_questions_json` с `non_flip_control` — вопросом, ответ на который правка не меняет (контроль).

## Карта ноутбука

- **§2** модель и общие функции (из `SPD-Faith Bench.ipynb`, с фиксами для Kaggle);
- **§3** данные: пары и проекция маски на image-токены;
- **§4** проход 1: по одному forward на каждую картинку → поведение, признаки для probing, подслои для §4.3;
- **§5** метод SPD-Faith (§4.3) на новом бенче — считается из кэша §4, без новых прогонов;
- **§6** linear probing;
- **§7** activation patching;
- **§8** связка с двумя режимами отказа из §4.1 статьи: *perceptual blindness* и *perception–reasoning dissociation*.

Каждая тяжёлая секция складывает результаты в `WORK_DIR`. Если сессия Kaggle оборвётся, повторный запуск
продолжит с места остановки.

**Первый запуск — smoke test:** `N_PAIRS = 16`, `PATCH_PAIRS_PER_CATEGORY = 2`. По строке tqdm будет видно
время на пример, после этого можно выставить полные значения.

## 0. Установка

**Kaggle:** справа включите *Accelerator → GPU T4 x2* и *Internet → On*. Затем добавьте MechFaithBench как датасет
(*Add Input → Datasets*). Для этого скачайте папку с Google Drive одним zip и загрузите её как приватный Kaggle
Dataset: `gdown --folder` не справится, у него лимит 50 файлов на папку.

**Локально:** `pip install -r ../requirements.txt`, релиз положите в `../data/MechFaithBench/`.

In [1]:
import os

IS_KAGGLE = os.path.exists("/kaggle/working")

if IS_KAGGLE:
    get_ipython().system('pip install -q -U "transformers>=4.45.0" "accelerate>=0.26.0"')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 95.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 81.7 MB/s eta 0:00:00:00:01


## 1. Конфигурация

In [2]:
import glob
import json

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

MODEL_PATH = "Qwen/Qwen2-VL-7B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def pick_dtype(device):
    """bf16 only on GPUs that support it natively (Ampere+). Kaggle's T4/P100 get fp16, CPU gets fp32."""
    if device != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16


DTYPE = pick_dtype(DEVICE)


DATA_DIR_OVERRIDE = "/kaggle/input/datasets/fluloeo/mechfaithbench/MechFaithBench_v1_9_final_2026-07-29"

def find_data_dir():
    """Папка релиза MechFaithBench (где лежит annotations.csv)."""
    if DATA_DIR_OVERRIDE:
        return DATA_DIR_OVERRIDE
    root = "/kaggle/input" if IS_KAGGLE else "../data"
    hits = sorted(os.path.join(d, f)
                  for d, _, files in os.walk(root, followlinks=True)
                  for f in files if f.lower() == "annotations.csv")
    if not hits:
        raise FileNotFoundError(
            f"annotations.csv не найден в {root}. Подключите датасет (Input -> Add Input) "
            f"или задайте DATA_DIR_OVERRIDE. Содержимое {root}: {os.listdir(root) if os.path.isdir(root) else 'нет'}")
    return os.path.dirname(hits[0])




DATA_DIR = find_data_dir()
WORK_DIR = "/kaggle/working/mechfaith" 
os.makedirs(WORK_DIR, exist_ok=True)

MAX_PIXELS = 640 * 640          # cap on the image size the model sees; MechFaithBench's 512x512 images become
                                # 504x504 -> an 18x18 grid of 324 image tokens
N_PAIRS = None                  # None = all 1600 pairs; smoke test: 16 (spread evenly over the 8 categories)
PATCH_PAIRS_PER_CATEGORY = 15   # pairs per category for activation patching (smoke test: 2)
PATCH_LAYER_STEP = 1            # 2 = patch every other layer, halves the patching time
SEED = 0

print(f"DATA_DIR = {DATA_DIR}")
print(f"WORK_DIR = {WORK_DIR}")
print(f"device = {DEVICE}, dtype = {DTYPE}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

DATA_DIR = /kaggle/input/datasets/fluloeo/mechfaithbench/MechFaithBench_v1_9_final_2026-07-29
WORK_DIR = /kaggle/working/mechfaith
device = cuda, dtype = torch.float16
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## 2. Модель и общие функции

Код хуков перенесён из `SPD-Faith Bench.ipynb` вместе со всеми фиксами, найденными при прогонах на Kaggle:

- `first_tensor`: в разных версиях `transformers` decoder-слой возвращает либо кортеж `(hidden_states, ...)`, либо
  голый тензор. Слепое `output[0]` на голом тензоре молча отрезает batch-размерность;
- при `device_map="auto"` на двух GPU вход слоя может лежать на GPU предыдущего слоя, поэтому переносим тензоры
  на устройство, где идёт вычисление;
- хуки живут на модулях, а не на Python-объекте, поэтому `attach()` сначала снимает чужие хуки, а `detach()` —
  свои.

**Поведенческий сигнал на весь ноутбук** — разность логитов на позиции ответа:
`LD = logsumexp(logits[yes-токены]) − logsumexp(logits[no-токены])`. `LD > 0` значит, что модель отвечает «yes».

In [ ]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_PATH, torch_dtype=DTYPE, device_map="auto" if DEVICE == "cuda" else None,
).eval()
processor = AutoProcessor.from_pretrained(MODEL_PATH, max_pixels=MAX_PIXELS)

cfg = model.config.text_config if hasattr(model.config, "text_config") else model.config
NUM_LAYERS = cfg.num_hidden_layers


def single_token_ids(words):
    """Token ids of those spellings that the tokenizer encodes as exactly one token."""
    ids = set()
    for w in words:
        enc = processor.tokenizer.encode(w, add_special_tokens=False)
        if len(enc) == 1:
            ids.add(enc[0])
    return sorted(ids)


YES_IDS = single_token_ids(["Yes", "yes", " Yes", " yes"])
NO_IDS = single_token_ids(["No", "no", " No", " no"])
assert YES_IDS and NO_IDS, (YES_IDS, NO_IDS)
print(f"{MODEL_PATH}: {NUM_LAYERS} layers | yes ids {YES_IDS} | no ids {NO_IDS}")

In [ ]:
def first_tensor(output):
    """A module's output as the hidden_states tensor, whether it returned a tuple or a bare tensor."""
    return output if isinstance(output, torch.Tensor) else output[0]


def get_decoder_layers(model):
    """The text decoder's layer stack. Its attribute path changed between transformers releases
    (model.model.layers vs model.model.language_model.layers), so try the known paths first."""
    for chain in (("model", "layers"), ("model", "language_model", "layers"),
                  ("language_model", "layers"), ("language_model", "model", "layers")):
        obj = model
        for attr in chain:
            obj = getattr(obj, attr, None)
            if obj is None:
                break
        if isinstance(obj, torch.nn.ModuleList):
            return obj
    raise AttributeError("decoder layers not found -- inspect model.named_modules()")


def get_image_token_id(model):
    """Id of the image placeholder token that the processor repeats once per image token."""
    for attr in ("image_token_id", "image_token_index"):
        val = getattr(model.config, attr, None)
        if val is not None:
            return val
    raise AttributeError("image token id not found on model.config")


LAYERS = get_decoder_layers(model)
IMAGE_TOKEN_ID = get_image_token_id(model)


def build_inputs(image, question):
    """One image + one question -> model inputs on DEVICE. The PIL image goes straight to the
    processor, so MAX_PIXELS alone decides its token grid (identical for both sides of a pair)."""
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return processor(text=[text], images=[image], return_tensors="pt").to(DEVICE)


def yes_no_logit_diff(logits_last):
    """LD = logsumexp(yes-token logits) - logsumexp(no-token logits) at the answer position, in fp32."""
    l = logits_last.float()
    return (torch.logsumexp(l[YES_IDS], 0) - torch.logsumexp(l[NO_IDS], 0)).item()


def answer_logits(inputs):
    """Next-token logits at the last position after a single forward pass (no generation)."""
    with torch.no_grad():
        return model(**inputs, use_cache=False).logits[0, -1]


class SublayerRecorder:
    """Read-side hooks from §4.3: for every decoder layer l they capture, in one forward pass,
      xl       -- the layer input (residual stream before the layer),
      xl_half  -- xl + MHA(LN(xl)), i.e. the residual stream between the two sublayers,
      xl_plus1 -- the layer output, xl_half + FFN(LN(xl_half)).
    """

    def __init__(self, layers):
        self.layers, self.hooks, self.state = layers, [], {}

    def attach(self):
        """Registers the hooks, after wiping any stale hooks left on these modules by earlier
        (possibly deleted) recorders -- `del recorder` does not remove hooks from the model."""
        self.detach()
        for layer in self.layers:
            layer._forward_pre_hooks.clear()
            layer._forward_hooks.clear()
            layer.self_attn._forward_hooks.clear()
        for i, layer in enumerate(self.layers):
            self.hooks.append(layer.register_forward_pre_hook(self._pre(i), with_kwargs=True))
            self.hooks.append(layer.self_attn.register_forward_hook(self._attn(i)))
            self.hooks.append(layer.register_forward_hook(self._out(i)))

    def detach(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []

    def reset(self):
        self.state = {i: {} for i in range(len(self.layers))}

    def _pre(self, i):
        def hook(module, args, kwargs):
            hs = args[0] if args else kwargs["hidden_states"]
            self.state[i]["xl"] = hs.detach()
        return hook

    def _attn(self, i):
        def hook(module, args, output):
            attn = first_tensor(output)
            xl = self.state[i]["xl"].to(attn.device)
            self.state[i]["xl"] = xl
            self.state[i]["xl_half"] = (xl + attn).detach()
        return hook

    def _out(self, i):
        def hook(module, args, output):
            self.state[i]["xl_plus1"] = first_tensor(output).detach()
        return hook

    def sublayer_cos(self, positions):
        """Per layer, mean over the sequence `positions` of cos(xl, xl_half) [MHA] and
        cos(xl_half, xl_plus1) [FFN], in fp32. Returns (mha, ffn), two lists of length NUM_LAYERS."""
        mha, ffn = [], []
        for i in range(len(self.layers)):
            s = self.state[i]
            dev = s["xl_half"].device
            pick = lambda t: t[:, positions.to(t.device)].float().to(dev)
            xl, xh, xp = pick(s["xl"]), pick(s["xl_half"]), pick(s["xl_plus1"])
            mha.append(F.cosine_similarity(xl, xh, dim=-1).mean().item())
            ffn.append(F.cosine_similarity(xh, xp, dim=-1).mean().item())
        return mha, ffn

## 3. Данные: пары и проекция маски на image-токены

- `load_pair`: обе стороны пары одного размера. Patching требует, чтобы позиции токенов совпадали 1:1, поэтому
  при разных размерах `edited` приводится к размеру `original` (такие случаи считаются в §4).
- `load_roi_mask`: попиксельное OR side-масок. `MASK_SCHEMA.md` требует брать маски, а не legacy union bbox.
  У пар `add`/`remove` одна сторона пустая по построению, и место правки задаёт маска другой стороны.
- `roi_token_mask`: Qwen2-VL ресайзит картинку до `(grid_h·14, grid_w·14)` и склеивает патчи 2×2, так что каждый
  image-токен — это клетка 28×28 пикселей, в порядке row-major. Порядок проверен на реальном препроцессоре.
  Токен считается ROI, если в его клетку попал хоть один пиксель маски.
- `token_groups`: позиции групп в последовательности. Они используются и для признаков, и для patching.

In [ ]:
ann = pd.read_csv(os.path.join(DATA_DIR, "annotations.csv"), keep_default_na=False)
if N_PAIRS is not None:
    per_category = -(-N_PAIRS // ann["category"].nunique())
    ann = ann.groupby("category").head(per_category)
ann = ann.reset_index(drop=True)
ANN = ann.set_index("record_id")

print(f"{len(ann)} pairs")
print(pd.DataFrame({
    "pairs": ann.groupby("category").size(),
    "gold (orig=yes, edit=no)": ann[ann.gold_original == "yes"].groupby("category").size(),
    "strict ROI eligible": ann[ann.roi_strict_spatial_eligible == "yes"].groupby("category").size(),
}).fillna(0).astype(int))


def load_pair(row):
    """(original, edited, resized): both sides as RGB images of the same size."""
    orig = Image.open(os.path.join(DATA_DIR, row.original_image)).convert("RGB")
    edit = Image.open(os.path.join(DATA_DIR, row.edited_image)).convert("RGB")
    resized = edit.size != orig.size
    if resized:
        edit = edit.resize(orig.size, Image.BICUBIC)
    return orig, edit, resized


def load_roi_mask(row, size):
    """Pixel-wise OR of the side-specific evidence masks as a bool array (H, W), or None."""
    union = None
    for col in ("roi_evidence_mask_original", "roi_evidence_mask_edited"):
        path = getattr(row, col)
        if not path:
            continue
        m = Image.open(os.path.join(DATA_DIR, path)).convert("L")
        if m.size != size:
            m = m.resize(size, Image.NEAREST)
        m = np.array(m) > 127
        union = m if union is None else (union | m)
    return union


def roi_token_mask(mask, grid_thw, merge=2, patch=14):
    """Bool vector over the image tokens (row-major over the (grid_h/2, grid_w/2) token grid):
    True where the token's 28x28 pixel cell contains at least one mask pixel."""
    _, gh, gw = (int(v) for v in grid_thw)
    cell = merge * patch
    th, tw = gh // merge, gw // merge
    m = Image.fromarray(mask.astype(np.uint8) * 255).resize((tw * cell, th * cell), Image.NEAREST)
    m = np.array(m) > 127
    return m.reshape(th, cell, tw, cell).any(axis=(1, 3)).reshape(-1)


def roi_tokens_for(inputs, mask):
    """roi_token_mask for these inputs' image grid; all False when the pair has no mask."""
    grid = inputs["image_grid_thw"][0]
    if mask is None:
        return np.zeros(int(grid.prod()) // 4, dtype=bool)
    return roi_token_mask(mask, grid)


def token_groups(inputs, roi_tok, rng=None):
    """Sequence positions (LongTensors on CPU) of:
      img      -- all image tokens;           roi / nonroi -- image tokens inside / outside the mask;
      question -- tokens after the image up to, not including, the last one (question + chat template);
      last     -- the answer position, whose logits give LD;
      placebo  -- (only if rng is given) random non-ROI image tokens, as many as ROI tokens."""
    ids = inputs["input_ids"][0].cpu()
    img = (ids == IMAGE_TOKEN_ID).nonzero(as_tuple=True)[0]
    roi_tok = torch.as_tensor(roi_tok, dtype=torch.bool)
    assert len(img) == len(roi_tok), (len(img), len(roi_tok))
    groups = {
        "img": img,
        "roi": img[roi_tok],
        "nonroi": img[~roi_tok],
        "question": torch.arange(int(img[-1]) + 1, len(ids) - 1),
        "last": torch.tensor([len(ids) - 1]),
    }
    if rng is not None:
        k = min(len(groups["roi"]), len(groups["nonroi"]))
        groups["placebo"] = groups["nonroi"][torch.as_tensor(np.sort(rng.choice(len(groups["nonroi"]), k, replace=False)))]
    return groups

**Проверка глазами.** Для трёх пар: `original`, `edited` и `edited` с подсвеченными ROI-токенами (красные клетки
28×28). Клетки должны лежать на месте правки. Если они смещены, дальше в ноутбуке ROI-анализ будет неверным.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

examples = ann[(ann.roi_evidence_mask_original != "") | (ann.roi_evidence_mask_edited != "")].head(3)
fig, axes = plt.subplots(len(examples), 3, figsize=(12, 4 * len(examples)), squeeze=False)
for ax_row, row in zip(axes, examples.itertuples()):
    orig, edit, _ = load_pair(row)
    mask = load_roi_mask(row, orig.size)
    inputs = build_inputs(orig, row.question)
    roi = roi_tokens_for(inputs, mask)
    _, gh, gw = (int(v) for v in inputs["image_grid_thw"][0])
    th, tw = gh // 2, gw // 2
    sx, sy = orig.size[0] / tw, orig.size[1] / th
    ax_row[0].imshow(orig); ax_row[0].set_title(f"original: gold={row.gold_original}")
    ax_row[1].imshow(edit); ax_row[1].set_title(f"edited: gold={row.gold_edited}")
    ax_row[2].imshow(edit); ax_row[2].set_title(f"{row.category}: {int(roi.sum())}/{len(roi)} ROI tokens")
    for k in np.flatnonzero(roi):
        r, c = divmod(k, tw)
        ax_row[2].add_patch(Rectangle((c * sx, r * sy), sx, sy, fill=False, edgecolor="red", linewidth=1))
    for ax in ax_row:
        ax.axis("off")
    print(f"{row.record_id}: {row.question}")
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, "roi_tokens_check.png"), dpi=120)
plt.show()

## 4. Проход 1: по одному forward на каждую картинку

Для каждой пары и каждой стороны — один прогон с основным `question`, без генерации. Сохраняем три вещи:

1. **поведение**: `LD` и ответ модели. Отдельно проверяем, что top-1 токен вообще «yes»/«no»;
2. **признаки для probing**: residual stream во всех 29 точках `hidden_states` (индекс 0 — эмбеддинги,
   `l` — выход decoder-слоя `l−1`, 28 — выход после финального RMSNorm) в трёх позициях: `last` (позиция ответа),
   среднее по `roi` и среднее по `nonroi` image-токенам. Хранение в fp16, около 1.2 МБ на пару;
3. **§4.3 Part B**: косинусы вход/выход MHA и FFN на позиции ответа и по ROI-токенам.

Результаты: `pass1/features/<record_id>.npy` и `pass1/meta.jsonl`. Уже посчитанные пары при перезапуске пропускаются.

In [ ]:
PASS1_DIR = os.path.join(WORK_DIR, "pass1")
FEAT_DIR = os.path.join(PASS1_DIR, "features")
META_PATH = os.path.join(PASS1_DIR, "meta.jsonl")
os.makedirs(FEAT_DIR, exist_ok=True)
POSITIONS = ("last", "roi", "nonroi")

recorder = SublayerRecorder(LAYERS)
recorder.attach()


def forward_side(image, question, mask):
    """One forward pass over one side of a pair -> (features [29, 3, D] fp16, info dict)."""
    inputs = build_inputs(image, question)
    g = token_groups(inputs, roi_tokens_for(inputs, mask))
    recorder.reset()
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, use_cache=False)
    hidden = out.hidden_states
    feats = np.zeros((len(hidden), len(POSITIONS), hidden[0].shape[-1]), dtype=np.float32)
    for l, hs in enumerate(hidden):
        for p, name in enumerate(POSITIONS):
            if len(g[name]):
                feats[l, p] = hs[0, g[name].to(hs.device)].float().mean(0).cpu().numpy()
    mha_last, ffn_last = recorder.sublayer_cos(g["last"])
    mha_roi, ffn_roi = recorder.sublayer_cos(g["roi"]) if len(g["roi"]) else ([], [])
    logits = out.logits[0, -1]
    info = {
        "ld": yes_no_logit_diff(logits),
        "top1_is_yes_no": int(logits.argmax().item() in YES_IDS + NO_IDS),
        "n_img_tokens": len(g["img"]),
        "n_roi_tokens": len(g["roi"]),
        "mha_last": mha_last, "ffn_last": ffn_last, "mha_roi": mha_roi, "ffn_roi": ffn_roi,
    }
    return np.clip(feats, -65000, 65000).astype(np.float16), info


def pass1_done():
    """Record ids with meta lines for both sides and a feature file on disk (a pair interrupted
    halfway is simply recomputed; duplicate meta lines are dropped when meta is loaded)."""
    if not os.path.exists(META_PATH):
        return set()
    with open(META_PATH) as f:
        sides = pd.DataFrame([json.loads(line) for line in f if line.strip()], columns=["record_id", "side"])
    both = sides.drop_duplicates().groupby("record_id").size()
    on_disk = {os.path.splitext(name)[0] for name in os.listdir(FEAT_DIR)}
    return set(both[both == 2].index) & on_disk


done = pass1_done()
for row in tqdm(list(ann.itertuples()), desc="pass 1"):
    if row.record_id in done:
        continue
    orig, edit, resized = load_pair(row)
    mask = load_roi_mask(row, orig.size)
    feats, infos = [], {}
    for side, image in (("original", orig), ("edited", edit)):
        f, infos[side] = forward_side(image, row.question, mask)
        feats.append(f)
    np.save(os.path.join(FEAT_DIR, f"{row.record_id}.npy"), np.stack(feats))
    with open(META_PATH, "a") as fh:
        for side in ("original", "edited"):
            fh.write(json.dumps({"record_id": row.record_id, "category": row.category, "side": side,
                                 "gold": getattr(row, f"gold_{side}"), "resized": resized,
                                 **infos[side]}) + "\n")

### 4.1 Поведение

Внутри пары ответы противоположные, поэтому каждая пара попадает ровно в одну группу:

- **correct_both** — модель права на обеих сторонах: правку «видит» и отвечает верно;
- **edit_blind** — одинаковый ответ на обе стороны, значит одна сторона обязательно неверна: модель *нечувствительна
  к правке*. Это аналог группы `blind` из SPD-Faith;
- **swapped** — ответы разные, но оба неверные.

Смещение данных, которое важно для §6: у 86% пар `gold_edited = yes`. Поэтому «ответ yes» и «это edited-картинка»
почти совпадают, и развести их помогают 224 пары с обратной полярностью (`gold_original = yes`).

In [ ]:
meta = pd.read_json(META_PATH, lines=True, dtype={"record_id": str})
meta = meta[meta.record_id.isin(pass1_done() & set(ann.record_id))]
meta = meta.drop_duplicates(["record_id", "side"], keep="last").reset_index(drop=True)
meta["pred"] = np.where(meta.ld > 0, "yes", "no")
meta["correct"] = meta.pred == meta.gold

wide = meta.pivot(index="record_id", columns="side", values=["pred", "correct", "ld", "n_roi_tokens"])
pairs = ANN.loc[wide.index, ["category", "gold_original", "gold_edited", "roi_strict_spatial_eligible"]].copy()
for side in ("original", "edited"):
    pairs[f"pred_{side}"] = wide[("pred", side)].astype(str)
    pairs[f"correct_{side}"] = wide[("correct", side)].astype(bool)
    pairs[f"ld_{side}"] = wide[("ld", side)].astype(float)
pairs["has_roi"] = wide[("n_roi_tokens", "original")].astype(int) > 0
pairs["reversed"] = pairs.gold_original == "yes"
pairs["group"] = np.select(
    [pairs.correct_original & pairs.correct_edited, pairs.pred_original == pairs.pred_edited],
    ["correct_both", "edit_blind"], default="swapped")

summary = pd.DataFrame({
    "pairs": pairs.groupby("category").size(),
    "image acc": meta.groupby("category").correct.mean().round(3),
    "pair acc": (pairs.group == "correct_both").groupby(pairs.category).mean().round(3),
    "edit_blind": (pairs.group == "edit_blind").groupby(pairs.category).mean().round(3),
    "yes rate": (meta.pred == "yes").groupby(meta.category).mean().round(3),
})
summary.loc["ALL"] = [len(pairs), meta.correct.mean().round(3), (pairs.group == "correct_both").mean().round(3),
                      (pairs.group == "edit_blind").mean().round(3), (meta.pred == "yes").mean().round(3)]
summary["pairs"] = summary["pairs"].astype(int)
print(summary)
print("\ngroups:", pairs.group.value_counts().to_dict())
print(f"top-1 token is yes/no: {meta.top1_is_yes_no.mean():.1%} | resized pairs: {int(meta.resized.sum() // 2)}"
      f" | pairs with ROI tokens: {int(pairs.has_roi.sum())}")

RECORD_IDS = list(pairs.index)
FEATS = np.stack([np.load(os.path.join(FEAT_DIR, f"{rid}.npy")) for rid in RECORD_IDS])  # [pairs, 2, 29, 3, D]
POS = {name: i for i, name in enumerate(POSITIONS)}
print("FEATS:", FEATS.shape, FEATS.dtype)

## 5. Метод SPD-Faith (§4.3) на MechFaithBench

Все величины уже посчитаны в проходе 1.

**Part A — дрейф представлений.** В SPD-Faith сравнивались два *вопроса* (same/different) на *одной* картинке.
Здесь сравниваются две *картинки* пары на *одном* вопросе: `CosSim_l(h(original, q), h(edited, q))`.

Интерпретация **обратная** SPD-Faith. Картинки отличаются причинной переменной, от которой зависит ответ, поэтому
у модели, которая правку видит, представления на позиции ответа *должны разойтись*. Высокое сходство у `edit_blind`
значит, что правка до решения не дошла. Кроме того, позиция ответа — это состояние *до* ответа, так что прежняя
претензия к §4.3 («сравниваем уже написанные разные тексты») здесь не применима.

**Part B — вклад подслоёв.** Как в SPD-Faith: `cos(x_l, x_{l+½})` для MHA и `cos(x_{l+½}, x_{l+1})` для FFN по
группам. Тезис статьи: FFN у unfaithful-примеров сильнее «поворачивает» представление. Здесь его аналог —
`edit_blind` против `correct_both`.

Полосы на графиках — 95% bootstrap-CI среднего по парам.

In [ ]:
GROUP_STYLE = {"correct_both": ("#2D5A96", "o-"), "edit_blind": ("#e4503f", "^--"), "swapped": ("#8a8a8a", "s:")}


def bootstrap_ci(arr, n_boot=500, seed=0):
    """95% bootstrap CI of the mean over rows of arr [n, points], resampling rows."""
    rng = np.random.default_rng(seed)
    means = np.stack([arr[rng.integers(0, len(arr), len(arr))].mean(0) for _ in range(n_boot)])
    return np.percentile(means, 2.5, axis=0), np.percentile(means, 97.5, axis=0)


def plot_curves(ax, curves, title, ylabel, styles=None, xlabel="layer", x=None):
    """curves: {label: array [n_samples, n_points]} -> mean and 95% bootstrap CI per label."""
    for label, arr in curves.items():
        arr = np.asarray(arr, dtype=float)
        if arr.ndim != 2 or len(arr) == 0:
            continue
        color, style = (styles or {}).get(label, (None, "o-"))
        x = np.arange(arr.shape[1]) if x is None else np.asarray(x)
        (line,) = ax.plot(x, arr.mean(0), style, color=color, markersize=3, linewidth=1.8,
                          label=f"{label} (n={len(arr)})")
        if len(arr) > 1:
            lo, hi = bootstrap_ci(arr)
            ax.fill_between(x, lo, hi, color=line.get_color(), alpha=0.15)
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)


def by_group(values, mask=None):
    """Split per-pair rows of `values` by the pair's behavioural group."""
    mask = np.ones(len(pairs), bool) if mask is None else np.asarray(mask)
    return {g: values[(pairs.group.values == g) & mask] for g in GROUP_STYLE}


def pair_cos(pos_name):
    """CosSim(original, edited) per pair and layer at one feature position -> [pairs, 29]."""
    a = FEATS[:, 0, :, POS[pos_name]].astype(np.float32)
    b = FEATS[:, 1, :, POS[pos_name]].astype(np.float32)
    return (a * b).sum(-1) / (np.linalg.norm(a, axis=-1) * np.linalg.norm(b, axis=-1) + 1e-6)


fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
plot_curves(axes[0], by_group(pair_cos("last")), "Part A: CosSim(original, edited) at the answer position",
            "cosine", GROUP_STYLE, xlabel="hidden_states index (0 = embeddings, 28 = after final norm)")
plot_curves(axes[1], by_group(pair_cos("roi"), pairs.has_roi), "Part A: same, mean over ROI tokens",
            "cosine", GROUP_STYLE, xlabel="hidden_states index")
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, "part_a.png"), dpi=150); plt.show()


def sublayer_by_group(key):
    """Per-pair average of the two sides' sublayer cosines (meta column `key`) -> {group: [n, 28]}."""
    per_image = meta.set_index(["record_id", "side"])[key]
    rows, keep = [], []
    for rid in RECORD_IDS:
        o, e = per_image[(rid, "original")], per_image[(rid, "edited")]
        keep.append(len(o) > 0)
        rows.append(np.mean([o, e], axis=0) if len(o) else np.full(NUM_LAYERS, np.nan))
    return by_group(np.array(rows), keep)


fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for ax, key, title in [(axes[0, 0], "mha_last", "MHA, answer position"), (axes[0, 1], "ffn_last", "FFN, answer position"),
                       (axes[1, 0], "mha_roi", "MHA, ROI tokens"), (axes[1, 1], "ffn_roi", "FFN, ROI tokens")]:
    plot_curves(ax, sublayer_by_group(key), f"Part B: {title}", "cos(sublayer input, output)", GROUP_STYLE,
                xlabel="decoder layer")
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, "part_b.png"), dpi=150); plt.show()

## 6. Linear probing

Зонд — `StandardScaler` + логистическая регрессия с L2 (`C = 0.1`). Отдельный зонд на каждую пару
«точка `hidden_states` × позиция». Оценка — **out-of-fold** точность по 5 фолдам `GroupKFold` с группировкой по паре:
обе стороны пары всегда в одном фолде. Иначе зонд запомнил бы сцену по одной стороне и был бы оценён на второй.

| зонд | признаки | цель | что проверяет |
|---|---|---|---|
| `answer@last` | позиция ответа | gold-ответ картинки (yes/no) | закодирован ли *правильный ответ* перед его выдачей |
| `edit@roi` | среднее по ROI-токенам | сторона (edited/original) | закодирована ли *сама правка* визуально |
| `edit@nonroi` | среднее по остальным image-токенам | сторона | **контроль**: ожидаем уровень случайности |

Если `edit@nonroi` заметно выше 0.5, правку можно извлечь из тех мест, где её нет. На пробных парах действительно
около 90% изменившихся пикселей лежат вне маски: это артефакты генерации (перерисованные волны и т.п.). Тогда
`edit@roi` нужно сравнивать с `edit@nonroi`, а не с 0.5, а patching (§7) покажет, влияют ли эти артефакты на ответ.

Линия `shuffled` — тот же зонд на перемешанных метках, то есть уровень случайности для данного объёма данных.
`answer@last` дополнительно разбит по полярности пары, чтобы отделить «ответ» от «какая это картинка» (см. §4.1).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

N_FOLDS = 5
PROBE_C = 0.1


def probe_oof(X, y, groups):
    """Out-of-fold P(y = 1) of a linear probe, folds grouped so a pair never straddles train and test."""
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=PROBE_C, max_iter=2000))
    cv = GroupKFold(min(N_FOLDS, len(np.unique(groups))))
    return cross_val_predict(clf, X, y, groups=groups, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]


n_pairs, n_points = FEATS.shape[0], FEATS.shape[2]
pair_of_image = np.repeat(np.arange(n_pairs), 2)                       # image order: pair 0 orig, pair 0 edit, ...
y_side = np.tile([0, 1], n_pairs)
y_answer = (np.stack([pairs.gold_original.values, pairs.gold_edited.values], 1) == "yes").reshape(-1).astype(int)
image_has_roi = np.repeat(pairs.has_roi.values, 2)

PROBES = {  # name: (feature position, target, images used)
    "answer@last": ("last", y_answer, np.ones(2 * n_pairs, bool)),
    "edit@roi": ("roi", y_side, image_has_roi),
    "edit@nonroi": ("nonroi", y_side, image_has_roi),
}

PROBE_PATH = os.path.join(WORK_DIR, "probing.npz")
if os.path.exists(PROBE_PATH) and np.load(PROBE_PATH)["record_ids"].tolist() == RECORD_IDS:
    saved = np.load(PROBE_PATH)
    OOF, OOF_SHUF, Y_SHUF = ({name: saved[f"{kind}_{name}"] for name in PROBES} for kind in ("oof", "shuf", "yshuf"))
    print("probing: loaded from cache")
else:
    rng = np.random.default_rng(SEED)
    OOF, OOF_SHUF, Y_SHUF = {}, {}, {}
    for name, (pos, y, use) in PROBES.items():
        Y_SHUF[name] = np.full(len(y), -1)
        Y_SHUF[name][use] = rng.permutation(y[use])         # chance-level control: same probe, labels shuffled
        OOF[name] = np.full((n_points, len(y)), np.nan)
        OOF_SHUF[name] = np.full((n_points, len(y)), np.nan)
        for l in tqdm(range(n_points), desc=name):
            X = FEATS[:, :, l, POS[pos]].reshape(2 * n_pairs, -1)[use].astype(np.float32)
            OOF[name][l, use] = probe_oof(X, y[use], pair_of_image[use])
            OOF_SHUF[name][l, use] = probe_oof(X, Y_SHUF[name][use], pair_of_image[use])
    np.savez(PROBE_PATH, record_ids=np.array(RECORD_IDS), **{f"oof_{k}": v for k, v in OOF.items()},
             **{f"shuf_{k}": v for k, v in OOF_SHUF.items()}, **{f"yshuf_{k}": v for k, v in Y_SHUF.items()})


def probe_acc(name, image_mask=None, shuffled=False):
    """Out-of-fold accuracy per hidden_states index, optionally on a subset of images."""
    _, y, use = PROBES[name]
    use = use if image_mask is None else use & image_mask
    oof, target = (OOF_SHUF[name], Y_SHUF[name]) if shuffled else (OOF[name], y)
    return ((oof[:, use] > 0.5) == target[use]).mean(1)


fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for name, color in zip(PROBES, ("#2D5A96", "#e4503f", "#8a8a8a")):
    axes[0].plot(probe_acc(name), "o-", color=color, markersize=3, label=name)
axes[0].plot(probe_acc("answer@last", shuffled=True), ":", color="black", label="shuffled (answer@last)")
axes[0].axhline(0.5, color="gray", linewidth=0.8)
axes[0].set(title="Linear probes: out-of-fold accuracy", xlabel="hidden_states index", ylabel="accuracy")
reversed_img = np.repeat(pairs.reversed.values, 2)
axes[1].plot(probe_acc("answer@last", ~reversed_img), "o-", markersize=3, label=f"typical pairs (edited = yes)")
axes[1].plot(probe_acc("answer@last", reversed_img), "^--", markersize=3, label=f"reversed pairs (edited = no)")
axes[1].axhline(0.5, color="gray", linewidth=0.8)
axes[1].set(title="answer@last by pair polarity (answer vs which image)", xlabel="hidden_states index",
            ylabel="accuracy")
for ax in axes:
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, "probing.png"), dpi=150); plt.show()

BEST = {name: int(np.nanargmax(probe_acc(name))) for name in PROBES}
image_category = np.repeat(pairs.category.values, 2)
print("best hidden_states index:", BEST)
print(pd.DataFrame({name: {cat: round(float(probe_acc(name, image_category == cat)[BEST[name]]), 3)
                           for cat in sorted(pairs.category.unique())} for name in PROBES}))

## 7. Activation patching

**Постановка (denoising).** Берём пару, где модель права на обеих сторонах. Прогон *clean* — `edited`, прогон
*corrupted* — `original`, вопрос один и тот же, ответы противоположные. В corrupted-прогоне на выходе decoder-слоя `l`
заменяем residual stream в группе позиций значениями из clean-прогона и пересчитываем всё, что идёт дальше. Метрика —
доля восстановленного поведения:

$$\text{recovery} = \frac{LD_{patch} - LD_{corr}}{LD_{clean} - LD_{corr}}$$

`1` значит, что модель отвечает как на clean, `0` — что патч ничего не изменил.

**Группы позиций:** `roi` (image-токены правки), `nonroi` (остальные image-токены), `placebo` (случайные не-ROI
токены в том же количестве, что ROI), `question` (текст после картинки), `last` (позиция ответа).

**Как читать результат.** На ранних слоях решает `roi`: информация о правке сидит там, где правка. На поздних
решает `last`: к позиции ответа информация уже перенесена attention-ом. Слой, где `roi` падает, а `last` растёт, —
это место переноса.

**Проверки.** Патч *всей* последовательности на слое 0 обязан дать recovery ≈ 1: это проверка корректности хука.
Для `non_flip_control`-вопроса («есть ли на картинке кайтсёрфер?»), на который правка не влияет, патч ROI не должен
заметно менять `LD`.

Результаты по каждой паре пишутся в `patching.jsonl` (перезапуск продолжает с места остановки).

In [ ]:
PATCH_PATH = os.path.join(WORK_DIR, "patching.jsonl")
PATCH_GROUPS = ("roi", "nonroi", "placebo", "question", "last")
PATCH_LAYERS = list(range(0, NUM_LAYERS, PATCH_LAYER_STEP))

recorder.detach()  # patching installs its own hooks; the read-side recorder is not needed any more


def run_and_cache(inputs):
    """Clean run: LD and the output of every decoder layer (the residual stream after layer l)."""
    cache = [None] * len(LAYERS)

    def make(i):
        def hook(module, args, output):
            cache[i] = first_tensor(output).detach()
        return hook

    handles = [layer.register_forward_hook(make(i)) for i, layer in enumerate(LAYERS)]
    try:
        ld = yes_no_logit_diff(answer_logits(inputs))
    finally:
        for h in handles:
            h.remove()
    return ld, cache


def run_patched(inputs, layer_idx, positions, source):
    """Corrupted run in which the output of decoder layer `layer_idx` at `positions` is replaced by
    `source` (that layer's clean output) before the rest of the network runs on top of it."""
    def hook(module, args, output):
        hs = first_tensor(output).clone()
        idx = positions.to(hs.device)
        hs[:, idx] = source[:, positions.to(source.device)].to(hs.device, hs.dtype)
        return hs if isinstance(output, torch.Tensor) else (hs,) + tuple(output[1:])

    handle = LAYERS[layer_idx].register_forward_hook(hook)
    try:
        return yes_no_logit_diff(answer_logits(inputs))
    finally:
        handle.remove()


def non_flip_question(row):
    for q in json.loads(row.diagnostic_questions_json or "[]"):
        if q.get("type") == "non_flip_control":
            return q["question"]
    return None


def patch_pair(row):
    orig, edit, _ = load_pair(row)
    mask = load_roi_mask(row, orig.size)
    clean_in, corr_in = build_inputs(edit, row.question), build_inputs(orig, row.question)
    assert torch.equal(clean_in["input_ids"], corr_in["input_ids"]), "token positions of the two sides differ"
    g = token_groups(corr_in, roi_tokens_for(corr_in, mask), rng=np.random.default_rng(SEED))

    ld_clean, cache = run_and_cache(clean_in)
    ld_corr = yes_no_logit_diff(answer_logits(corr_in))
    everything = torch.arange(corr_in["input_ids"].shape[1])
    result = {
        "record_id": row.Index, "category": row.category, "n_roi": len(g["roi"]),
        "ld_clean": ld_clean, "ld_corr": ld_corr, "layers": PATCH_LAYERS,
        "ld_full_layer0": run_patched(corr_in, 0, everything, cache[0]),
        "ld_patch": {name: [run_patched(corr_in, l, g[name], cache[l]) for l in PATCH_LAYERS]
                     for name in PATCH_GROUPS},
    }

    q_nf = non_flip_question(row)
    if q_nf:
        nf_clean_in, nf_corr_in = build_inputs(edit, q_nf), build_inputs(orig, q_nf)
        nf_ld_clean, nf_cache = run_and_cache(nf_clean_in)
        nf_roi = token_groups(nf_corr_in, roi_tokens_for(nf_corr_in, mask))["roi"]
        result["non_flip"] = {
            "question": q_nf, "ld_clean": nf_ld_clean, "ld_corr": yes_no_logit_diff(answer_logits(nf_corr_in)),
            "ld_roi": [run_patched(nf_corr_in, l, nf_roi, nf_cache[l]) for l in PATCH_LAYERS],
        }
    return result


eligible = pairs[(pairs.group == "correct_both") & (pairs.roi_strict_spatial_eligible == "yes") & pairs.has_roi]
rng = np.random.default_rng(SEED)
PATCH_IDS = []
for cat, d in eligible.groupby("category"):
    PATCH_IDS += list(rng.choice(d.index.values, size=min(len(d), PATCH_PAIRS_PER_CATEGORY), replace=False))
print(f"patching {len(PATCH_IDS)} pairs x {len(PATCH_LAYERS)} layers x {len(PATCH_GROUPS)} position groups")
if not PATCH_IDS:
    print("нет пар, где модель права на обеих сторонах и есть ROI -- patching нечего считать")

done = set()
if os.path.exists(PATCH_PATH):
    with open(PATCH_PATH) as f:
        done = {json.loads(line)["record_id"] for line in f if line.strip()}
for rid in tqdm(PATCH_IDS, desc="patching"):
    if rid in done:
        continue
    res = patch_pair(next(ANN.loc[[rid]].itertuples()))
    with open(PATCH_PATH, "a") as fh:
        fh.write(json.dumps(res) + "\n")

In [ ]:
assert os.path.exists(PATCH_PATH), "patching.jsonl не найден -- сначала выполните ячейку выше"
patch = pd.read_json(PATCH_PATH, lines=True, dtype={"record_id": str})
patch = patch[patch.record_id.isin(PATCH_IDS)].drop_duplicates("record_id", keep="last").reset_index(drop=True)
assert len(patch), "в patching.jsonl нет пар из PATCH_IDS -- сначала выполните ячейку выше"
denom = (patch.ld_clean - patch.ld_corr).values
full = (patch.ld_full_layer0 - patch.ld_corr).values / denom
print(f"{len(patch)} pairs | sanity, whole sequence patched at layer 0: recovery "
      f"min {full.min():.3f} / max {full.max():.3f} (must be ~1)")

RECOVERY = {name: (np.stack([np.array(v[name]) for v in patch.ld_patch]) - patch.ld_corr.values[:, None])
                  / denom[:, None] for name in PATCH_GROUPS}

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
plot_curves(axes[0], RECOVERY, "Activation patching (clean = edited -> corrupted = original)",
            "recovery", xlabel="patched decoder layer", x=PATCH_LAYERS)
axes[0].axhline(0, color="gray", linewidth=0.8)
im = axes[1].imshow(np.stack([RECOVERY[n].mean(0) for n in PATCH_GROUPS]), aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set(yticks=range(len(PATCH_GROUPS)), yticklabels=PATCH_GROUPS, title="mean recovery",
            xticks=range(len(PATCH_LAYERS)), xticklabels=PATCH_LAYERS, xlabel="patched decoder layer")
axes[1].tick_params(axis="x", labelsize=7)
plt.colorbar(im, ax=axes[1])
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, "patching.png"), dpi=150); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
for ax, name in zip(axes[:2], ("roi", "last")):
    plot_curves(ax, {cat: RECOVERY[name][(patch.category == cat).values] for cat in sorted(patch.category.unique())},
                f"recovery by category: {name}", "recovery", xlabel="patched decoder layer", x=PATCH_LAYERS)
nf = patch[patch.non_flip.notna()]
flip_delta = np.abs(np.stack([np.array(v["roi"]) for v in nf.ld_patch]) - nf.ld_corr.values[:, None])
nf_delta = np.abs(np.stack([np.array(v["ld_roi"]) - v["ld_corr"] for v in nf.non_flip]))
plot_curves(axes[2], {"flip question": flip_delta, "non-flip control": nf_delta},
            "control: |LD change| after patching ROI", "|LD_patch - LD_corr|",
            xlabel="patched decoder layer", x=PATCH_LAYERS)
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, "patching_by_category.png"), dpi=150); plt.show()

## 8. Связка с faithfulness: blindness против dissociation

В §4.1 статьи есть два режима отказа, и их различают по картам внимания/TAM. Здесь то же самое делается зондом, и
делается количественно. Для каждой картинки сравниваем:

- **видит ли модель правку**: out-of-fold предсказание зонда `edit@roi` на лучшем слое;
- **говорит ли она правильно**: ответ модели.

| зонд `edit@roi` | ответ модели | режим |
|---|---|---|
| ✓ | ✓ | faithful |
| ✓ | ✗ | **dissociation**: правка закодирована, но ответ её не отражает |
| ✗ | ✗ | **perceptual blindness**: правка не закодирована |
| ✗ | ✓ | угадывание |

Для dissociation-картинок дальше смотрим точность зонда `answer@last` по слоям: где ответ ещё извлекается и где
теряется. Это прямая проверка тезиса статьи о дрейфе в глубоких слоях.

Оговорка: если `edit@nonroi` в §6 тоже высокий, зонд `edit@roi` мог опираться на артефакты, и «видит» здесь
означает «представление отличает стороны».

In [ ]:
best = BEST["edit@roi"]
sees = (OOF["edit@roi"][best] > 0.5) == y_side
says = np.stack([pairs.correct_original.values, pairs.correct_edited.values], 1).reshape(-1).astype(bool)
use = image_has_roi
mode = np.select([sees & says, sees & ~says, ~sees & ~says], ["faithful", "dissociation", "blindness"], "guessing")

names = dict(rownames=["category"], colnames=["mode"], margins=True)
table = pd.crosstab(image_category[use], mode[use], normalize="index", **names).round(3)
table["n"] = pd.crosstab(image_category[use], mode[use], **names)["All"]
print(f"edit@roi probe at hidden_states index {best}:")
print(table)

fig, ax = plt.subplots(figsize=(8, 4.5))
for m, style in (("faithful", "o-"), ("dissociation", "^--"), ("blindness", "s:")):
    sel = use & (mode == m)
    if sel.sum():
        ax.plot(probe_acc("answer@last", sel), style, markersize=3, label=f"{m} (n={int(sel.sum())})")
ax.axhline(0.5, color="gray", linewidth=0.8)
ax.set(title="Is the correct answer decodable at the answer position?", xlabel="hidden_states index",
       ylabel="answer@last probe accuracy")
ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, "faithfulness_modes.png"), dpi=150); plt.show()

## 9. Сохранение сводки

Сырьё уже лежит в `WORK_DIR`: `pass1/`, `probing.npz`, `patching.jsonl`, картинки. Здесь одним JSON собраны числа
для отчёта.

In [ ]:
report = {
    "model": MODEL_PATH, "dtype": str(DTYPE), "pairs": len(pairs), "max_pixels": MAX_PIXELS,
    "behaviour": summary.reset_index().rename(columns={"index": "category"}).to_dict(orient="records"),
    "groups": pairs.group.value_counts().to_dict(),
    "probing": {name: {"accuracy": probe_acc(name).round(4).tolist(), "best_index": BEST[name]} for name in PROBES},
    "patching": {"pairs": len(patch), "layers": PATCH_LAYERS,
                 "mean_recovery": {n: RECOVERY[n].mean(0).round(4).tolist() for n in PATCH_GROUPS}},
    "faithfulness_modes": pd.Series(mode[use]).value_counts().to_dict(),
}
with open(os.path.join(WORK_DIR, "report.json"), "w") as f:
    json.dump(report, f, ensure_ascii=False, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print(f"saved {os.path.join(WORK_DIR, 'report.json')}")